# 用于LR++实验的验证notebook

In [ ]:
import os
import pandas as pd
import json
import logging

import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    f1_score,
    auc
)

import torch
import pickle
import argparse
import numpy as np
from tqdm import tqdm

import sys
sys.path.append(os.path.dirname(os.getcwd()))
from utils.helper import (
    setup_logging,
    preprocess_combined_data,
    clean_feature_names,
    jload,
    jdump
)

import warnings
warnings.filterwarnings("ignore")

def ks_score(true_labels, predicted_probabilities):
    """
    Calculate the KS score.
    """
    fpr, tpr, _ = roc_curve(true_labels, predicted_probabilities)
    return max(tpr - fpr)

def calculate_metrics(true_labels, predicted_probabilities):
    """
    Calculate and return evaluation metrics for a model.
    Metrics include accuracy, ROC AUC, PR AUC, F1 score, KS score, and more.
    """
    # Convert predicted probabilities to binary labels (threshold = 0.5)
    predicted_labels = [1 if probability > 0.5 else 0 for probability in predicted_probabilities]

    # Calculate Precision-Recall curve and PR AUC
    precision, recall, _ = precision_recall_curve(true_labels, predicted_probabilities)
    pr_auc = auc(recall, precision)

    # Collect all metrics in a dictionary
    metrics = {
        'accuracy': accuracy_score(true_labels, predicted_labels),
        'ROC_AUC': roc_auc_score(true_labels, predicted_probabilities),
        'PR_AUC': pr_auc,
        'F1_score': f1_score(true_labels, predicted_labels),
        'KS_score': ks_score(true_labels, predicted_probabilities),
        'num': len(true_labels),
    }

    return metrics

def plot_roc(test_label, predicted_probabilities, model_name):
    fpr, tpr, _ = roc_curve(test_label, predicted_probabilities)

    ks_statistics = max(tpr - fpr)
    ks_x = fpr[np.argmax(tpr - fpr)]
    ks_y = tpr[np.argmax(tpr - fpr)]

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'ROC curve (area = {metrics["ROC_AUC"]:.3f})')
    plt.plot([0, 1], [0, 1], 'k--')

    plt.plot([ks_x, ks_x], [ks_x, ks_y], 'r--', linewidth=1.5)
    plt.text(ks_x + 0.01, ks_x + (ks_y - ks_x) / 2 + 0.02, f'KS={ks_statistics:.3f}', fontsize=12)

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic for {model_name}')
    plt.legend(loc="lower right")
    plt.show()

def save_model(model, path, desc, metrics):
    model_info = {
        'model': model,
        'desc': desc,
        'metrics': metrics
    }
    with open(path, "wb") as file:
        pickle.dump(model_info, file)

def load_model(path):
    with open(path, "rb") as file:
        model_info = pickle.load(file)
    return model_info


In [ ]:
folder = "/data2/youxiang/repos/RiskReasoner/datasets_new/woe_boxings_bins"
training_data = "train_filtered_woe_bins.parquet"
test_data = "test_filtered_woe_bins.parquet"
oot = "oot_filtered_woe_bins.parquet"

features_bins = [
    'vara242_woe',
    'vara225_woe',
    'varb744_woe',
    'varb746_woe',
    'varb94_woe',
    'varb783_woe',
    'varc3_woe',
    'varc940_woe',
    'varc798_woe'
]

training_data = pd.read_parquet(os.path.join(folder, training_data))
testing_data = pd.read_parquet(os.path.join(folder, test_data))
oot_data = pd.read_parquet(os.path.join(folder, oot))

training_data = training_data[features_bins + ["target"]]
testing_data = testing_data[features_bins + ["target"]]
oot_data = oot_data[features_bins + ["target"]]

In [ ]:
LABEL = "target"  # Global variable for the target column
feature_columns = [col for col in training_data.columns if col != LABEL]

model_info = load_model("/data2/youxiang/repos/RiskReasoner/datasets_new/woe_boxings_bins/lr_woe")
lr_model = model_info["model"]

In [ ]:
# Calculate metrics on Test
predicted_probabilities = lr_model.predict_proba(testing_data[feature_columns])[:, 1]
metrics = calculate_metrics(
    true_labels=testing_data[LABEL],
    predicted_probabilities=predicted_probabilities,
)
metrics

plot_roc(testing_data[LABEL], predicted_probabilities, "Logistic Regression")
metrics

In [ ]:
# Calculate metrics on OOT
predicted_probabilities = lr_model.predict_proba(oot_data[feature_columns])[:, 1]
metrics = calculate_metrics(
    true_labels=oot_data[LABEL],
    predicted_probabilities=predicted_probabilities,
)
metrics

plot_roc(oot_data[LABEL], predicted_probabilities, "Logistic Regression")
metrics

In [ ]:
coefs = lr_model.coef_[0]
intercept = lr_model.intercept_[0]

from kan.compiler import kanpiler
import sympy as sp

x_symbols = sp.symbols('x0:' + str(len(feature_columns)))
linear_comb = sum(coefs[i] * x_symbols[i] for i in range(len(feature_columns))) + intercept
logistic_expr = 1 / (1 + sp.exp(-linear_comb))

kan_model = kanpiler(x_symbols, logistic_expr, grid=10, k=5, auto_save=False)

kan_model(X_train)
kan_model.plot()

In [ ]:
for name, p in kan_model.named_parameters():
    if "mask" in name:
        p.requires_grad = True

X_train = torch.tensor(training_data[feature_columns].values, dtype=torch.float32)
y_train = torch.tensor(training_data[LABEL].values, dtype=torch.float32).unsqueeze(1)
batch_size = 1024
loader = DataLoader(
    TensorDataset(X_train, y_train), 
    batch_size=batch_size, 
    shuffle=True)
X_test = torch.tensor(testing_data[feature_columns].values, dtype=torch.float32)
X_oot = torch.tensor(oot_data[feature_columns].values, dtype=torch.float32)

optimizer = torch.optim.Adam(kan_model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = torch.nn.functional.binary_cross_entropy_with_logits

for name, p in kan_model.named_parameters():
    print(name, p.shape, p.requires_grad)

In [ ]:
def get_kan_metrics(data="test"):
    kan_model.eval()

    if data == "test":
        with torch.no_grad():
            y_prob = torch.sigmoid(kan_model(X_test)).numpy().ravel()
            y_pred = (y_prob > 0.5).astype(int)
            true_labels = testing_data[LABEL]

    elif data == "oot":
        with torch.no_grad():
            y_prob = torch.sigmoid(kan_model(X_oot)).numpy().ravel()
            y_pred = (y_prob > 0.5).astype(int)
            true_labels = oot_data[LABEL]

    kan_metrics = calculate_metrics(
        true_labels=true_labels,
        predicted_probabilities=y_prob,
    )
    kan_model.train()
    return kan_metrics, y_prob

all_test_metrics = []
all_oot_metrics = []

for epoch in range(30):
    total_loss = 0
    for x, y in tqdm(loader, desc="Batches:"):
        # kan_model.update_grid_from_samples(x)
        y_pred = kan_model(x)
        loss = criterion(y_pred, y)
        optimizer.zero_grad()
        loss.backward()
        # add this with caution
        torch.nn.utils.clip_grad_norm_(kan_model.parameters(), max_norm=1.0)
        optimizer.step()

        # FIXME: use this somehow cause the performance to drop
        # for n, p in kan_model.named_parameters():
        #     if "scale_sp" in n or "scale_base" in n:
        #         p.data.clamp(min=1e-20)

        total_loss += loss.item() * x.size(0)
        
    # for name, p in kan_model.named_parameters():
    #     if p.requires_grad:
    #         print(f"{name}: [{p.min().item():.4f}, {p.max().item():.4f}]")
    print(f"Epoch {epoch+1}, Loss: {total_loss / len(loader.dataset):.4f}")

    test_metrics, y_prob_test = get_kan_metrics(data="test")
    all_test_metrics.append(test_metrics)

    oot_metrics, y_prob_oot = get_kan_metrics(data="oot")
    all_oot_metrics.append(oot_metrics)

In [ ]:
pd.DataFrame(all_test_metrics)["KS_score"].plot(x="epochs", y="ks", title="KS on Test data", legend=True)

In [ ]:
pd.DataFrame(all_test_metrics)["KS_score"].plot(x="epochs", y="ks", title="KS on Test data", legend=True)

In [ ]:
lib = ['x', 'x^2', 'x^3', 'x^4', 'exp', 'log', 'sqrt', 'tanh', 'sin', 'abs']
kan_model.auto_symbolic(lib=lib)

fomular = kan_model.symbolic_formula()[0]
features_left = kan_model.symbolic_formula()[1]

fomular[0]